

# 📊 JSOM Smart Advisor – Course Data Processing

**Project:** JSOM Smart Advisor & Digital Ecosystem Optimization
**University:** Naveen Jindal School of Management, The University of Texas at Dallas

## Purpose

This Google Colab notebook is used to process course data collected from JSOM sources. The dataset is initially organized in a spreadsheet format and exported as a CSV file. Using Python in Google Colab, the CSV data is transformed into a structured JSON dataset.

The generated JSON file will serve as the course knowledge base for the JSOM Smart Advisor system. This allows the AI advisor to understand course details such as prerequisites, credit hours, and skills taught in each course.

## Why Google Colab

Google Colab provides a cloud-based Python environment that allows efficient data processing without installing additional software. It enables quick dataset uploads, easy Python execution, and fast transformation of spreadsheet data into JSON format.

## Data Processing Pipeline

```
JSOM Course Data (Google Sheets / Excel)
                ↓
           CSV Export
                ↓
        Google Colab (Python)
                ↓
     Data Cleaning & Transformation
                ↓
        Structured JSON Dataset
                ↓
      JSOM Smart Advisor System
```






## 🚀 How to Use This Notebook

1. Run the **first code cell** to upload the CSV file containing the course dataset.
2. In the **second code cell**, update the file name with your uploaded CSV file name.
3. Run the **second code cell** to convert the dataset into JSON format.

After running the notebook, the JSON file will be generated automatically and can be used as the course dataset for the JSOM Smart Advisor system.

In [18]:
from google.colab import files
uploaded = files.upload()

Saving JSOM_Smart_Advisor_Course_Dataset.csv to JSOM_Smart_Advisor_Course_Dataset (1).csv


In [19]:
import pandas as pd
import json
import re

df = pd.read_csv("JSOM_Smart_Advisor_Course_Dataset.csv")

# remove duplicate rows
df = df.drop_duplicates()

# remove extra whitespace from all string columns
df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

df.head()


/tmp/ipykernel_258/1884840810.py:11: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)


,course_id,title,credits,course_type,description,Prerequisites,OnlyOneOfThese,skills_taught
0,BUAN 6333,Foundations of Programming for Business Analytics,3.0,Elective,This course will introduce students to two of ...,There are no particular prerequisites for this...,NaN,"Python, R"
1,BUAN 6340,Programming for Data Science,3.0,Elective,"In the era of big data, it is critical to util...","BUAN 6356, BUAN 6383, MIS 6386, MIS 6334, MIS ...",NaN,"Python, Programming Fundamentals, Data Manipul..."
2,BUAN 6346,Big Data,3.0,Elective,Master the rapidly evolving Big Data landscape...,"BUAN 6320, MIS 6320, MIS 6326.",NaN,"Big Data, Hadoop, HDFS, Impala, Hive, Data Mod..."
3,BUAN 6347,Advanced Big Data Analytics,3.0,Elective,The course covers Spark using Scala in a Hadoo...,BUAN 6346,NaN,"scala programming, spark programming, big data..."
4,BUAN 6358,AWS Cloud Analytics,3.0,Elective,"This course integrates data engineering, machi...",NaN,NaN,"data engineering, data pipeline development, d..."


In [20]:
df.count()

,0
course_id,82
title,82
credits,82
course_type,82
description,81
Prerequisites,46
OnlyOneOfThese,34
skills_taught,79


In [21]:
import re
import pandas as pd

def parse_prerequisites(prereq):

    # treat blank / none as no prerequisites
    if pd.isna(prereq) or str(prereq).strip().lower() in ["", "none"]:
        return None

    prereq = str(prereq)

    # normalize separators
    prereq = prereq.replace(";", " or ")
    prereq = prereq.replace(",", " or ")

    # remove extra spaces
    prereq = re.sub(r'\s+', ' ', prereq)

    groups = []

    # CASE 1: parentheses groups exist
    parentheses_groups = re.findall(r'\((.*?)\)', prereq)

    if parentheses_groups:
        for group in parentheses_groups:

            courses = re.findall(r'[A-Z]{4}\s?\d{3,4}', group.upper())

            if courses:
                groups.append(list(set(courses)))

    else:
        # CASE 2: no parentheses → split by AND
        and_groups = re.split(r'\band\b', prereq, flags=re.IGNORECASE)

        for group in and_groups:

            courses = re.findall(r'[A-Z]{4}\s?\d{3,4}', group.upper())

            if courses:
                groups.append(list(set(courses)))

    return groups if groups else None

In [24]:
df["prerequisites_clean"] = df["Prerequisites"].apply(parse_prerequisites)

In [25]:
def parse_only_one(row):

    value = row["OnlyOneOfThese"]
    course_id = row["course_id"]

    if pd.isna(value) or value.strip() == "":
        return None

    courses = re.findall(r'[A-Z]{4}\s?\d{4}', value.upper())

    if course_id not in courses:
        courses.append(course_id)

    return list(set(courses))

In [26]:
df["only_one_of_these_clean"] = df.apply(parse_only_one, axis=1)

In [27]:
def parse_skills(skills):

    if pd.isna(skills):
        return []

    skills = re.split(r'[;,]', skills)

    cleaned = [s.strip() for s in skills if s.strip() != ""]

    return cleaned

In [28]:
df["skills_clean"] = df["skills_taught"].apply(parse_skills)

In [ ]:
df = df.dropna(subset=["course_id"])

In [29]:
courses = []

for _, row in df.iterrows():

    course = {
        "course_id": row["course_id"],
        "title": row["title"],
        "credits": row["credits"],
        "course_type": row["course_type"],
        "description": row["description"],
        "prerequisites": row["prerequisites_clean"],
        "only_one_of_these": row["only_one_of_these_clean"],
        "skills_taught": row["skills_clean"]
    }

    courses.append(course)

In [31]:
with open("courses.json", "w") as f:
    json.dump(courses, f, indent=4)

In [32]:
files.download("courses.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>